In [0]:
%sql
CREATE TABLE IF NOT EXISTS retail_lakehouse.silver.sales (
  order_id STRING,
  order_date DATE,
  customer_id STRING,
  product_id STRING,
  quantity INT,
  unit_price DOUBLE,
  updated_at TIMESTAMP,
  total_amount DOUBLE,
  ingestion_time TIMESTAMP
);

In [0]:
from pyspark.sql.functions import col, to_date, expr
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

bronze_sales = spark.table("retail_lakehouse.bronze.sales")

clean_df = (
    bronze_sales
    .filter(col("order_id").isNotNull())
    .filter(col("customer_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .withColumn("order_date", to_date(col("order_date")))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("unit_price", col("unit_price").cast("double"))
    .withColumn("total_amount", col("quantity") * col("unit_price"))
)

window_spec = Window.partitionBy("order_id").orderBy(col("updated_at").desc())

dedup_df = (
    clean_df
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
display(dedup_df)

In [0]:
dedup_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("retail_lakehouse.silver.sales")

In [0]:
%sql
SELECT last_watermark_value
FROM retail_lakehouse.audit.watermark_control
WHERE pipeline_id = 'sales_pipeline';

In [0]:
from pyspark.sql.functions import lit
from datetime import datetime

# Extract watermark value from the SQL query result
watermark_result = _sqldf.collect()
if watermark_result:
    last_watermark_value = watermark_result[0][0]
else:
    # No watermark exists yet - use epoch time to process all data
    last_watermark_value = datetime(1970, 1, 1)

# Filter cleaned and deduplicated data for incremental processing
incremental_df = dedup_df.filter(col("updated_at") > lit(last_watermark_value))

In [0]:
%sql
MERGE INTO retail_lakehouse.audit.watermark_control t
USING (
  SELECT 'sales_pipeline' AS pipeline_id, max(updated_at) AS last_watermark_value
  FROM retail_lakehouse.silver.sales
) s
ON t.pipeline_id = s.pipeline_id
WHEN MATCHED THEN UPDATE SET t.last_watermark_value = s.last_watermark_value
WHEN NOT MATCHED THEN INSERT *